# Qwen trên Colab T4 — GGUF Q4_K_M và local API

**MODEL-SERVE-01 · STATIC VALIDATED — NOT GPU TESTED**

Chạy lần lượt bốn phần: cấu hình → cache → server → tests. Notebook dành cho
thử nghiệm tương tác trong Colab; chưa xác nhận inference, VRAM hoặc context trên GPU.
Q4_K_M là k-quant 4-bit, không phải AWQ INT4. Chỉ tải một model text, không vision encoder.
Qwen3.5 có kiến trúc hybrid; không áp dụng công thức KV-cache của Qwen3.
Qwen3-14B/16K có thể thiếu VRAM trên T4. Nếu OOM: chọn 8192, có thể giảm GPU_LAYERS,
rồi chạy lại thủ công; không tự đổi weights/quantization hoặc retry.

Nguồn đã đối chiếu 11/11/2026:
[Qwen3.5 GGUF](https://huggingface.co/axteam/Qwen3.5-9B-GGUF),
[Qwen3 GGUF](https://huggingface.co/Qwen/Qwen3-14B-GGUF),
[llama.cpp v0.9.0/server](https://github.com/ggml-org/llama.cpp/blob/cd959bae99ff4bed22bef2604f45241c09bbb8c8/tools/server/README.md),
[kiến trúc backend](https://github.com/ggml-org/llama.cpp/blob/cd959bae99ff4bed22bef2604f45241c09bbb8c8/src/llama-arch.cpp),
[giới hạn Colab](https://research.google.com/colaboratory/faq.html).
Không tunnel, public URL, keep-alive hoặc thay runtime/Gateway DACNTT.

## PART 1 — Cấu hình và GPU

Trong Colab: **Runtime → Change runtime type → T4 GPU** nếu được cấp.
Không có GPU thì dừng trước khi tải model. GPU khác được báo rõ và build theo compute capability thực tế.
Chạy lại cell cấu hình sẽ dừng server do notebook sở hữu. Drive mount có thể yêu cầu đăng nhập Google.

In [ ]:
import hashlib
import json
import os
import platform
import re
import secrets
import shutil
import signal
import socket
import struct
import subprocess
import time
from pathlib import Path

import pandas as pd
import requests
from IPython.display import display

# fmt: off
#@title Cấu hình và kiểm tra GPU
MODEL_CHOICE = "Qwen3.5-9B" #@param ["Qwen3.5-9B", "Qwen3-14B"]
CONTEXT_SIZE = 16384 #@param [8192, 16384] {type:"raw"}
USE_GOOGLE_DRIVE = True #@param {type:"boolean"}
SAVE_MODEL_TO_DRIVE = True #@param {type:"boolean"}
ENABLE_THINKING = False #@param {type:"boolean"}
PORT = 8000 #@param {type:"integer"}
GPU_LAYERS = "Auto" #@param {type:"string"}
# Auto yêu cầu offload tối đa; số nguyên >0 cho phép giảm offload thủ công.

# fmt: on

ROOT = Path("/content/dacntt_qwen")
ROOT.mkdir(parents=True, exist_ok=True)
STATE = ROOT / "server.json"


def process_identity(pid):
    try:
        # starttime tránh dừng nhầm PID được tái sử dụng sau kernel restart.
        return Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[19]
    except FileNotFoundError:
        return None


def stop_server():
    global SERVER_READY
    SERVER_READY = False
    if not STATE.exists():
        return
    state = json.loads(STATE.read_text())
    pid = state["pid"]
    if state["starttime"] is not None and process_identity(pid) == state["starttime"]:
        command = Path(f"/proc/{pid}/cmdline").read_bytes().split(b"\0")
        if (
            str(state["binary"]).encode() not in command
            or str(state["model"]).encode() not in command
            or os.getpgid(pid) != pid
        ):
            raise RuntimeError("Không xác minh được process sở hữu; hãy Restart Session.")
        os.killpg(pid, signal.SIGTERM)
        for _ in range(50):
            proc = globals().get("SERVER_PROCESS")
            if proc is not None and proc.pid == pid:
                proc.poll()  # reap child, kể cả zombie
            if process_identity(pid) is None:
                break
            time.sleep(0.1)
        else:
            if process_identity(pid) == state["starttime"]:
                os.killpg(pid, signal.SIGKILL)
        proc = globals().get("SERVER_PROCESS")
        if proc is not None and proc.pid == pid:
            proc.wait(timeout=10)
    STATE.unlink(missing_ok=True)
    print("STOPPED: server của notebook.")


stop_server()
assert MODEL_CHOICE in {"Qwen3.5-9B", "Qwen3-14B"}
assert CONTEXT_SIZE in {8192, 16384} and 1024 <= PORT <= 65535
OFFLOAD = 999 if GPU_LAYERS == "Auto" else int(GPU_LAYERS)
assert OFFLOAD > 0, "Không cho phép CPU-only; dùng số GPU layers >0."

MODELS = {
    "Qwen3.5-9B": {
        "repo": "axteam/Qwen3.5-9B-GGUF",
        "revision": "557d48d08797c03394db2877c14a76d094def3ed",
        "filename": "Qwen3.5-9B-Q4_K_M.gguf",
        "size": 5780090752,
        "sha256": "11d4468d09fe020a4d5212a60b8119dfaa1a6f26cf80c138fa6af87fb43ba449",
        "folder": "Qwen3.5-9B-Q4_K_M",
        "alias": "qwen3.5-9b-t4",
    },
    "Qwen3-14B": {
        "repo": "Qwen/Qwen3-14B-GGUF",
        "revision": "530227a7d994db8eca5ab5ced2fb692b614357fd",
        "filename": "Qwen3-14B-Q4_K_M.gguf",
        "size": 9001752960,
        "sha256": "500a8806e85ee9c83f3ae08420295592451379b4f8cf2d0f41c15dffeb6b81f0",
        "folder": "Qwen3-14B-Q4_K_M",
        "alias": "qwen3-14b-t4",
    },
}
MODEL = MODELS[MODEL_CHOICE].copy()


def gpu_memory():
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free,compute_cap",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        timeout=15,
        check=True,
    )
    fields = [s.strip() for s in result.stdout.splitlines()[0].split(",")]
    return dict(
        zip(["name", "total_MiB", "used_MiB", "free_MiB", "compute_cap"], fields, strict=True)
    )


try:
    GPU = gpu_memory()
except (OSError, subprocess.SubprocessError) as exc:
    raise RuntimeError("ERROR: không có NVIDIA GPU khả dụng. Đổi runtime sang GPU/T4.") from exc
print("GPU:", GPU)
print("CUDA driver: NVIDIA device hiện diện; backend CUDA sẽ được kiểm tra tại PART 3.")
print("CUDA toolkit nvcc:", shutil.which("nvcc") or "MISSING")
print("Runtime:", platform.platform(), "Python", platform.python_version())
print("Local disk GiB:", round(shutil.disk_usage("/content").free / 2**30, 2))
if "T4" not in GPU["name"]:
    print("CẢNH BÁO: GPU không phải T4; kết quả chỉ áp dụng cho GPU thực tế ở trên.")
CUDA_ARCH = GPU["compute_cap"].replace(".", "")
print("Model:", MODEL_CHOICE, "| GGUF Q4_K_M | Context yêu cầu:", CONTEXT_SIZE)

## PART 2 — Google Drive Model Cache

Cache: `/content/drive/MyDrive/DACNTT/models/<model>-Q4_K_M/`.
Xác minh kích thước, GGUF header và SHA-256 LFS của revision đã pin; không chỉ kiểm tra tên.
Copy weights về `/content/models/` trước khi load. Drive có thể không báo quota tài khoản
chính xác qua filesystem: vẫn kiểm tra dung lượng, bắt lỗi copy/quota và tiếp tục với local.
Cache hỏng được giữ nguyên đến khi có bản tải hoàn chỉnh thay thế. Không đụng model còn lại.
SHA được tính lại khi tái sử dụng nên có thể mất vài phút đọc Drive.

In [ ]:
# @title Chuẩn bị model — Drive hoặc download fallback
stop_server()
LOCAL_DIR = Path("/content/models") / MODEL["folder"]
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = LOCAL_DIR / MODEL["filename"]
MARGIN = 512 * 2**20


def validate_model(path):
    if not path.is_file() or path.stat().st_size != MODEL["size"]:
        return False
    with path.open("rb") as stream:
        magic, version, tensors, metadata = struct.unpack("<4sIQQ", stream.read(24))
        if magic != b"GGUF" or version != 3 or tensors == 0 or metadata == 0:
            return False
        stream.seek(0)
        digest = hashlib.file_digest(stream, "sha256").hexdigest()
    return digest == MODEL["sha256"]


def require_space(directory, size):
    free = shutil.disk_usage(directory).free
    print(f"Disk {directory}: free={free / 2**30:.2f} GiB; cần={(size + MARGIN) / 2**30:.2f} GiB")
    if free < size + MARGIN:
        raise OSError("Không đủ disk space; giải phóng bộ nhớ hoặc Restart Session.")


def verified_copy(source, destination):
    require_space(destination.parent, MODEL["size"])
    temporary = destination.with_suffix(".copying")
    try:
        shutil.copyfile(source, temporary)
        if not validate_model(temporary):
            raise ValueError("Copy không khớp GGUF header/size/SHA-256.")
        os.replace(temporary, destination)
    finally:
        temporary.unlink(missing_ok=True)


DRIVE_DIR = None
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/DACNTT/models") / MODEL["folder"]
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Drive filesystem free GiB:", round(shutil.disk_usage(DRIVE_DIR).free / 2**30, 2))
    except Exception as exc:
        print("ERROR: Drive mount/cache không khả dụng:", type(exc).__name__, "→ dùng local.")
        DRIVE_DIR = None

try:
    if validate_model(MODEL_PATH):
        print("READY: local cache đã xác minh.")
    else:
        found_in_drive = False
        if DRIVE_DIR is not None:
            try:
                cached = DRIVE_DIR / MODEL["filename"]
                found_in_drive = validate_model(cached)
                if found_in_drive:
                    print("FOUND_IN_DRIVE:", cached.name)
                    verified_copy(cached, MODEL_PATH)
                    print("COPIED_TO_LOCAL:", MODEL_PATH)
            except OSError as exc:
                print(
                    "ERROR: không đọc/copy được Drive:", type(exc).__name__, "→ download fallback."
                )
                found_in_drive = False
        if not found_in_drive:
            require_space(LOCAL_DIR, MODEL["size"])
            print("DOWNLOADING:", MODEL["repo"], MODEL["filename"], "GGUF Q4_K_M")
            url = f"https://huggingface.co/{MODEL['repo']}/resolve/{MODEL['revision']}/{MODEL['filename']}"
            partial = MODEL_PATH.with_suffix(".downloading")
            deadline = time.monotonic() + 3600
            try:
                # Public repositories: không cần HF token, không in redirect URL/credentials.
                with requests.get(url, stream=True, timeout=(20, 120)) as response:
                    if response.status_code != 200:
                        raise RuntimeError(
                            f"Hugging Face HTTP {response.status_code}; không đổi checkpoint."
                        )
                    written = 0
                    with partial.open("wb") as stream:
                        for chunk in response.iter_content(8 * 2**20):
                            if time.monotonic() > deadline:
                                raise TimeoutError("Download quá 3600 giây; chạy lại thủ công.")
                            written += len(chunk)
                            if written > MODEL["size"]:
                                raise ValueError("Download vượt kích thước manifest.")
                            stream.write(chunk)
                if not validate_model(partial):
                    raise ValueError("GGUF download không khớp header/size/SHA-256.")
                os.replace(partial, MODEL_PATH)
            finally:
                partial.unlink(missing_ok=True)
    if DRIVE_DIR is not None and SAVE_MODEL_TO_DRIVE:
        try:
            cached = DRIVE_DIR / MODEL["filename"]
            if not validate_model(cached):
                verified_copy(MODEL_PATH, cached)
                print("Đã lưu model hoàn chỉnh lên Drive.")
        except OSError as exc:
            print(
                "ERROR: Drive thiếu quota/không ghi được:",
                type(exc).__name__,
                "→ tiếp tục bằng local.",
            )
    print("READY:", MODEL_PATH, "| verified SHA-256:", MODEL["sha256"])
except Exception:
    print("ERROR: chuẩn bị model thất bại; không start server. Xem traceback.")
    raise

## PART 3 — Build CUDA và Start Model & API Server

Build source pin **v0.9.0 / cd959bae99ff4bed22bef2604f45241c09bbb8c8**, CUDA SM75 trên T4.
Cell build không load weights. Binary hợp lệ được dùng lại trong session.
Giới hạn build 20 phút/lệnh; readiness 10 phút. Log nằm trong `/content/dacntt_qwen/`.
Concurrency=1, chỉ text; giữ mặc định cache của backend, không suy đoán hybrid KV-cache.
Non-thinking dùng `--reasoning off`, request `reasoning_effort=none` và
`chat_template_kwargs.enable_thinking=false` được backend này hỗ trợ cho cả hai lựa chọn;
test còn kiểm tra có final content, không reasoning ngoài ý muốn.

In [ ]:
# @title Build llama-server CUDA (dùng lại binary nếu hợp lệ)
LLAMA_COMMIT = "cd959bae99ff4bed22bef2604f45241c09bbb8c8"
SOURCE = ROOT / "llama.cpp"
BINARY = SOURCE / "build/bin/llama-server"
BUILD_STAMP = ROOT / "build.json"
BUILD_LOG = ROOT / "build.log"
expected_build = {"commit": LLAMA_COMMIT, "cuda_arch": CUDA_ARCH}


def run_build(command, timeout=1200):
    print("Lệnh:", " ".join(map(str, command)))
    with BUILD_LOG.open("a") as log:
        try:
            subprocess.run(
                list(map(str, command)),
                stdout=log,
                stderr=subprocess.STDOUT,
                timeout=timeout,
                check=True,
            )
        except (subprocess.SubprocessError, OSError):
            print(BUILD_LOG.read_text(errors="replace")[-6000:])
            raise RuntimeError(f"Build FAILED; lệnh trên, log {BUILD_LOG}") from None


def cuda_binary_ok():
    if not BINARY.is_file():
        return False
    result = subprocess.run(
        [str(BINARY), "--list-devices"], capture_output=True, text=True, timeout=30
    )
    return result.returncode == 0 and bool(re.search(r"CUDA\d", result.stdout + result.stderr))


if not (
    BUILD_STAMP.exists()
    and json.loads(BUILD_STAMP.read_text()) == expected_build
    and cuda_binary_ok()
):
    stop_server()
    if not shutil.which("nvcc"):
        raise RuntimeError("CUDA toolkit nvcc MISSING. Chọn Colab GPU runtime và Restart Session.")
    BUILD_LOG.write_text("")
    run_build(["apt-get", "update"], 300)
    run_build(["apt-get", "install", "-y", "build-essential", "cmake", "ninja-build", "git"], 600)
    if not SOURCE.exists():
        run_build(["git", "init", SOURCE], 60)
        run_build(
            [
                "git",
                "-C",
                SOURCE,
                "remote",
                "add",
                "origin",
                "https://github.com/ggml-org/llama.cpp.git",
            ],
            60,
        )
    run_build(["git", "-C", SOURCE, "fetch", "--depth", "1", "origin", LLAMA_COMMIT], 300)
    run_build(["git", "-C", SOURCE, "checkout", "--detach", LLAMA_COMMIT], 60)
    run_build(
        [
            "cmake",
            "-S",
            SOURCE,
            "-B",
            SOURCE / "build",
            "-G",
            "Ninja",
            "-DGGML_CUDA=ON",
            f"-DCMAKE_CUDA_ARCHITECTURES={CUDA_ARCH}",
            "-DCMAKE_BUILD_TYPE=Release",
            "-DLLAMA_CURL=OFF",
            "-DLLAMA_OPENSSL=OFF",
            "-DLLAMA_BUILD_TESTS=OFF",
            "-DLLAMA_BUILD_EXAMPLES=OFF",
            "-DLLAMA_BUILD_SERVER=ON",
        ]
    )
    run_build(["cmake", "--build", SOURCE / "build", "--target", "llama-server", "-j", "2"])
    if not cuda_binary_ok():
        raise RuntimeError("FAILED: binary không liệt kê CUDA device; không chạy CPU fallback.")
    BUILD_STAMP.write_text(json.dumps(expected_build))
print("CUDA backend/device: verified qua --list-devices; chưa chứng minh GPU inference.")
print(subprocess.run([str(BINARY), "--version"], capture_output=True, text=True, timeout=15).stdout)

In [ ]:
# @title Start Server — readiness và API probe có timeout
stop_server()
if not validate_model(MODEL_PATH) or not cuda_binary_ok():
    raise RuntimeError("Chạy PART 2 và build CUDA thành công trước.")
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", PORT)) == 0:
        raise RuntimeError("Port đang có process khác; không dừng process lạ. Đổi PORT ở PART 1.")

API_KEY = secrets.token_urlsafe(32)  # Chỉ trong RAM/env subprocess, không hiển thị.
API = f"http://127.0.0.1:{PORT}"
HEADERS = {"Authorization": "Bearer " + API_KEY}
SERVER_LOG = ROOT / "server.log"
command = [
    str(BINARY),
    "--model",
    str(MODEL_PATH),
    "--alias",
    MODEL["alias"],
    "--host",
    "127.0.0.1",
    "--port",
    str(PORT),
    "--ctx-size",
    str(CONTEXT_SIZE),
    "--parallel",
    "1",
    "--n-gpu-layers",
    str(OFFLOAD),
    "--jinja",
    "--reasoning",
    "on" if ENABLE_THINKING else "off",
    "--reasoning-format",
    "deepseek",
    "--batch-size",
    "512",
    "--ubatch-size",
    "128",
]
child_env = {
    k: v for k, v in os.environ.items() if not k.startswith("LLAMA_ARG_") and k != "LLAMA_API_KEY"
}
child_env["LLAMA_API_KEY"] = API_KEY


def api_json(method, route, payload=None, timeout=30):
    response = requests.request(
        method, API + route, headers=HEADERS, json=payload, timeout=(10, timeout)
    )
    if not response.ok:
        # Response của server cục bộ; key luôn được redact trước output.
        detail = response.text[:1500].replace(API_KEY, "<REDACTED>")
        raise RuntimeError(f"HTTP {response.status_code} {route}: {detail}")
    return response.json()


def chat_payload(prompt, max_tokens=256, **extra):
    payload = {
        "model": MODEL["alias"],
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": max_tokens,
        "temperature": 0,
        "stream": False,
        "chat_template_kwargs": {"enable_thinking": ENABLE_THINKING},
    }
    if not ENABLE_THINKING:
        payload["reasoning_effort"] = "none"
    payload.update(extra)
    return payload


def final_content(data):
    message = data["choices"][0]["message"]
    content = message.get("content")
    if not isinstance(content, str) or not content.strip():
        raise ValueError("Không có final content; có thể chỉ trả reasoning hoặc hết output budget.")
    if not ENABLE_THINKING and (
        message.get("reasoning_content")
        or message.get("reasoning")
        or "<think>" in content
        or "</think>" in content
    ):
        raise ValueError("Non-thinking chưa đạt: response chứa reasoning/thinking tags.")
    return content


try:
    with SERVER_LOG.open("w") as log:
        SERVER_PROCESS = subprocess.Popen(
            command, stdout=log, stderr=subprocess.STDOUT, env=child_env, start_new_session=True
        )
    STATE.write_text(
        json.dumps(
            {
                "pid": SERVER_PROCESS.pid,
                "starttime": process_identity(SERVER_PROCESS.pid),
                "binary": str(BINARY),
                "model": str(MODEL_PATH),
            }
        )
    )
    deadline = time.monotonic() + 600
    for _ in range(300):
        if SERVER_PROCESS.poll() is not None:
            raise RuntimeError(f"llama-server exited code={SERVER_PROCESS.returncode}.")
        try:
            health = requests.get(API + "/health", headers=HEADERS, timeout=2)
            if health.status_code == 200:
                break
        except requests.RequestException:
            pass
        if time.monotonic() >= deadline:
            raise TimeoutError("Readiness quá 600 giây.")
        time.sleep(2)
    else:
        raise TimeoutError("Readiness hết số lần kiểm tra.")
    log_text = SERVER_LOG.read_text(errors="replace")
    offloaded = re.findall(r"offloaded\s+(\d+)/(\d+)\s+layers", log_text)
    if not offloaded or int(offloaded[-1][0]) <= 0:
        raise RuntimeError("Chưa xác nhận offload >0 từ log; không báo GPU READY.")
    models_response = api_json("GET", "/v1/models")
    if MODEL["alias"] not in [m["id"] for m in models_response.get("data", [])]:
        raise ValueError("/v1/models không trả alias đã chọn.")
    props = api_json("GET", "/props")
    actual_context = props.get("default_generation_settings", {}).get("n_ctx")
    if actual_context != CONTEXT_SIZE or props.get("total_slots") != 1:
        raise ValueError(f"Context/slots khác yêu cầu: {actual_context}/{props.get('total_slots')}")
    probe_response = api_json(
        "POST",
        "/v1/chat/completions",
        chat_payload("Chỉ trả lời một từ: Xin chào", 128),
        timeout=180,
    )
    print("API probe final content:", final_content(probe_response))
    SERVER_READY = True
    ACTIVE_CONFIG = (MODEL_CHOICE, CONTEXT_SIZE, ENABLE_THINKING, PORT, GPU_LAYERS)
    print(
        f"MODEL SERVER\nStatus: READY\nModel: {models_response['data'][0]['id']}"
        f"\nQuantization: GGUF Q4_K_M (SHA-256 verified)\nContext: {actual_context}"
        f"\nGPU: {gpu_memory()['name']}\nOffloaded layers: {offloaded[-1]}\nAPI: {API}/v1"
    )
    print("Local API chỉ trong Colab runtime; External API: UNAVAILABLE.")
except BaseException:
    stop_server()
    print("MODEL SERVER\nStatus: FAILED")
    tail = SERVER_LOG.read_text(errors="replace")[-7000:] if SERVER_LOG.exists() else ""
    print(tail.replace(API_KEY, "<REDACTED>"))
    if re.search(r"out of memory|cudaMalloc.*fail|CUDA.*alloc.*fail", tail, re.I):
        print("CUDA OOM: chọn CONTEXT_SIZE=8192; có thể giảm GPU_LAYERS. Không tự retry.")
    raise

### Kết nối Codex / DACNTT

**Local API** chỉ gọi được từ notebook/Colab runtime này. `127.0.0.1` trên máy Codex
không trỏ đến Colab. **External API: UNAVAILABLE**, không có URL bên ngoài được xác minh.
Chưa thể cấu hình Gateway trên máy Codex dùng notebook Colab Free này.
Adapter Gateway DACNTT thuộc task khác sau khi inference được kiểm chứng.

Format dưới đây là **mẫu chưa dùng được**, không phải endpoint public:

```env
LLM_PROVIDER=colab_qwen
MODEL_API_BASE_URL=<actual-reachable-url>/v1
MODEL_API_KEY=<configured-secret>
MODEL_NAME=qwen3.5-9b-t4
```

Khi chọn Qwen3-14B, MODEL_NAME tương ứng là `qwen3-14b-t4`.
API key local được sinh ngẫu nhiên trong RAM khi start; requests trong notebook tự dùng key.
Không in key/HF credential, không copy secrets vào output. Chỉ điền URL external sau khi
có hosting phù hợp và xác minh trực tiếp từ máy Codex trong task sau.

## PART 4 — Test API

Tests chạy tuần tự. PASS là kiểm tra API/engineering hẹp, chưa phải chất lượng tài chính
hay độ phù hợp với schemas DACNTT. Mỗi request có timeout; nếu timeout sẽ dừng server
và dừng các test còn lại để tránh request cũ tiếp tục chiếm GPU.
Long context tắt mặc định. Khi bật, dùng `/apply-template` + `/tokenize` đếm prompt đã
render bằng tokenizer thật: khoảng 8K (8192-context dành 512 output + 128 margin nên
prompt thực tế tối đa 7552), thêm khoảng 12K nếu context 16K.
Không lấy output tokens / tổng latency làm generation tokens/second.

In [ ]:
# fmt: off
#@title Run Tests
RUN_LONG_CONTEXT = False #@param {type:"boolean"}
# fmt: on
if not globals().get("SERVER_READY"):
    raise RuntimeError("Start Server thành công trước khi test.")
if ACTIVE_CONFIG != (MODEL_CHOICE, CONTEXT_SIZE, ENABLE_THINKING, PORT, GPU_LAYERS):
    raise RuntimeError("Cấu hình đã đổi; chạy lại PART 1 → PART 3.")


def check_json(content):
    value = json.loads(content)  # không strip reasoning hoặc markdown để che lỗi
    expected = {"metric": "gross_margin", "numerator": 100, "denominator": 1000, "percentage": 10}
    if not isinstance(value, dict) or not expected.keys() <= value.keys():
        raise ValueError("Thiếu JSON keys.")
    if value["metric"] != expected["metric"] or any(
        type(value[k]) not in (int, float) or value[k] != expected[k]
        for k in ("numerator", "denominator", "percentage")
    ):
        raise ValueError("JSON sai giá trị hoặc kiểu số.")


def check_basic(content):
    if "hà nội" not in content.casefold():
        raise ValueError("Không tìm thấy đáp án Hà Nội.")


def check_financial(content):
    if not re.search(r"(?<![\d.,])20(?:[.,]0+)?\s*%", content):
        raise ValueError("Không tìm thấy kết quả 20%.")
    if not all(number in content for number in ("100", "120")) or not re.search(
        r"[-−]|trừ", content
    ):
        raise ValueError("Thiếu công thức với hai giá trị nguồn.")


rows = []


def run_test(name, payload, checker):
    row = {
        "Test": name,
        "Status": "FAIL",
        "Input tokens": "UNKNOWN",
        "Output tokens": "UNKNOWN",
        "Latency s": "UNKNOWN",
        "Generation tok/s": "UNKNOWN",
        "GPU used MiB": "UNKNOWN",
        "GPU free MiB": "UNKNOWN",
        "Error": "",
    }
    started = time.perf_counter()
    try:
        data = api_json("POST", "/v1/chat/completions", payload, timeout=600)
        usage = data.get("usage") or {}
        row["Input tokens"] = usage.get("prompt_tokens", "UNKNOWN")
        row["Output tokens"] = usage.get("completion_tokens", "UNKNOWN")
        row["Generation tok/s"] = (data.get("timings") or {}).get("predicted_per_second", "UNKNOWN")
        content = final_content(data)
        print(name, "→", content[:2000])
        if data["choices"][0].get("finish_reason") == "length":
            raise ValueError("Output bị cắt vì hết token budget.")
        checker(content)
        row["Status"] = "PASS"
    except requests.Timeout:
        row["Error"] = "Request timeout; server đã dừng, không tiếp tục inference."
        stop_server()
    except Exception as exc:
        row["Error"] = str(exc).replace(API_KEY, "<REDACTED>")[:1500]
    finally:
        row["Latency s"] = round(time.perf_counter() - started, 3)
        try:
            memory = gpu_memory()
            row["GPU used MiB"], row["GPU free MiB"] = memory["used_MiB"], memory["free_MiB"]
        except (OSError, subprocess.SubprocessError):
            pass
        rows.append(row)


tests = [
    (
        "Vietnamese QA",
        chat_payload("Thủ đô của Việt Nam là thành phố nào? Trả lời ngắn gọn."),
        check_basic,
    ),
    (
        "Structured JSON",
        chat_payload(
            'Chỉ trả JSON, không markdown: metric="gross_margin", numerator=100, denominator=1000, '
            "percentage=10. Không thêm giải thích.",
            response_format={"type": "json_object"},
        ),
        check_json,
    ),
    (
        "Financial prompt",
        chat_payload(
            "Doanh thu thuần tăng từ 100 tỷ lên 120 tỷ đồng. Hãy tính tốc độ tăng trưởng phần trăm, "
            "nêu công thức và kết quả."
        ),
        check_financial,
    ),
]
for name, payload, checker in tests:
    run_test(name, payload, checker)
    if not SERVER_READY:
        break


def rendered_token_count(prompt):
    payload = chat_payload(prompt, 512)
    rendered = api_json("POST", "/apply-template", payload)["prompt"]
    return len(api_json("POST", "/tokenize", {"content": rendered, "add_special": True})["tokens"])


def long_prompt(target):
    reserve, margin = 512, 128
    target = min(target, CONTEXT_SIZE - reserve - margin)
    prefix = "Dữ liệu lặp để kiểm tra context, không phải dữ liệu tài chính thật.\n"
    suffix = "\nChỉ trả lời mã kết thúc: DACNTT_OK"
    unit = "Thông tin thử nghiệm trung tính. "
    low, high, best = 0, target, None
    for _ in range(16):  # binary search hữu hạn, không inference
        if low > high:
            break
        count = (low + high) // 2
        prompt = prefix + unit * count + suffix
        tokens = rendered_token_count(prompt)
        if tokens <= target:
            best = prompt, tokens
            low = count + 1
        else:
            high = count - 1
    if best is None or best[1] < target - 128:
        raise ValueError("Không dựng được prompt gần token target.")
    assert best[1] + reserve + margin <= CONTEXT_SIZE
    print("Long context actual rendered tokens:", best[1], "output reserve:", reserve)
    return best[0]


def check_long(content):
    if "DACNTT_OK" not in content:
        raise ValueError("Thiếu mã kết thúc; chưa đạt long-context smoke.")


if RUN_LONG_CONTEXT and SERVER_READY:
    for target in [8000, 12000] if CONTEXT_SIZE == 16384 else [8000]:
        try:
            prompt = long_prompt(target)
            run_test(f"Long context ~{target}", chat_payload(prompt, 512), check_long)
        except Exception as exc:
            rows.append(
                {
                    "Test": f"Long context ~{target}",
                    "Status": "FAIL",
                    "Error": str(exc).replace(API_KEY, "<REDACTED>")[:1500],
                }
            )
        if not SERVER_READY:
            break
elif not RUN_LONG_CONTEXT:
    print("Long context: SKIPPED (tắt mặc định).")
display(pd.DataFrame(rows).fillna("UNKNOWN"))
print(
    "VRAM là snapshot sau request, không phải peak. UNKNOWN = backend không báo hoặc chưa đo được."
)
print("Kết quả này không xác nhận chất lượng tài chính, full context hoặc schemas DACNTT phức tạp.")

### Stop Server, đổi model và Restart Session

Chạy cell dưới khi kết thúc hoặc trước khi đổi model. Đổi MODEL_CHOICE tại PART 1,
chạy lại tuần tự PART 1 → PART 4; build dùng lại nếu hợp lệ. Model còn lại trên Drive được giữ nguyên.
Để reset sạch: **Runtime → Disconnect and delete runtime**, kết nối lại, chọn GPU rồi chạy từ đầu.
Local `/content` và key tạm sẽ mất; Drive cache vẫn còn. Không có cơ chế giữ phiên sống.
Nếu lỗi: gửi traceback cùng đoạn cuối build.log/server.log đã kiểm tra không có thông tin riêng tư.
Không gửi secrets hoặc toàn bộ environment. Chưa mở vòng sửa tiếp trước khi có log Colab thực tế.

In [ ]:
# @title Stop Server
stop_server()